In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List
import math
import re

coverage_new = pd.read_csv("coverage_new.tsv", sep="\t")

In [ ]:
def extract_id(name: str) -> str:
    """
    Extracts the sample ID from the column name.
    Parameters
    name : str
        Column name, for example: "sample_01sib_normal" or "control"

    Returns
    str
        Extracted identifier (for example, "01sib") or the original name,
        if the pattern is not found.
    """
    m = re.search(r"_(\d+sib)_", name)
    return m.group(1) if m else name

In [ ]:
def zscore_from_coverage(
    df: pd.DataFrame, ref_samples: List[str] = None
) -> pd.DataFrame:
    """
    Calculate z-scores for coverage data using reference samples.

    Normalizes coverage by autosomal mean, then computes z-score for each sample
    relative to reference samples.

    Parameters
    df : pd.DataFrame
        DataFrame with 'chr', 'start' columns and sample columns with coverage values.
    ref_samples : list of str, optional
        List of reference sample IDs.
        If None, uses default.

    Returns
    pd.DataFrame
        DataFrame with 'chr', 'start' columns and z-score columns named '{sample}_z'.
    """
    # Default reference samples
    if ref_samples is None:
        ref_samples = ["47sib", "33sib", "44sib", "7sib", "39sib"]

    meta_cols = ["chr", "start"]
    sample_cols = [c for c in df.columns if c not in meta_cols]

    numeric = df[sample_cols].astype(float)

    # Normalize by autosomes
    auto_mask = ~df["chr"].isin(["chrX", "X", "chrY", "Y"])
    auto_mean = numeric.loc[auto_mask].mean(axis=0)

    norm = numeric.div(auto_mean, axis=1)

    z_df = pd.DataFrame(index=df.index)
    ref_set = set(ref_samples)

    for sample in sample_cols:

        ref_cols = [c for c in sample_cols if extract_id(c) in ref_set and c != sample]

        if len(ref_cols) < 2:
            continue

        mean_ref = norm[ref_cols].mean(axis=1)
        std_ref = norm[ref_cols].std(axis=1)

        std_ref = std_ref.replace(0, np.nan)

        z = (norm[sample] - mean_ref) / std_ref

        z_df[f"{sample}_z"] = z

    return pd.concat([df[meta_cols].copy(), z_df], axis=1)

In [4]:
z_new = zscore_from_coverage(coverage_new)

In [ ]:
def add_genome_pos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add genome position column for cumulative coordinates across chromosomes.
    Converts chromosome names to numeric order, removes non-standard chromosomes,
    sorts by chromosome and start, then calculates cumulative genome position.

    Parameters
    df : pd.DataFrame
        DataFrame with 'chr' and 'start' columns.

    Returns
    pd.DataFrame
        DataFrame with added 'genome_pos' column, sorted by chromosome and start.
        Only includes autosomes 1-22 and sex chromosomes X, Y.
    """
    df = df.copy()

    df["chr"] = df["chr"].astype(str).str.replace("chr", "", regex=False)

    chrom_order = [str(i) for i in range(1, 23)] + ["X", "Y"]

    df = df[df["chr"].isin(chrom_order)]

    df["chr"] = pd.Categorical(df["chr"], categories=chrom_order, ordered=True)

    df = df.sort_values(["chr", "start"])

    offsets = df.groupby("chr")["start"].max().cumsum().shift(fill_value=0)

    df["genome_pos"] = df.apply(
        lambda row: row["start"] + offsets.get(row["chr"], 0), axis=1
    )

    return df

In [6]:
z_new = add_genome_pos(z_new)

In [ ]:
def plot_in_grids(
    z_df: pd.DataFrame, out_dir: str = "plots_compare", per_page: int = 4
) -> None:
    """
    Plot z-score distributions in grid pages.
    Creates multi-page grids of scatter plots showing z-scores across genome
    positions. Each page contains up to 'per_page' plots arranged in a 2x2 grid.

    Parameters
    z_df : pd.DataFrame
        DataFrame with 'genome_pos' column and z-score columns ending with '_z'.
    out_dir : str, default="plots_compare"
        Output directory for saved plot pages.
    per_page : int, default=4
        Number of plots per page (max 4 for 2x2 grid).

    Returns
    None
        Saves plot pages to out_dir and closes them.
    """
    out = Path(out_dir)
    out.mkdir(exist_ok=True)

    sample_cols = [c for c in z_new.columns if c.endswith("_z")]

    n_pages = math.ceil(len(sample_cols) / per_page)

    for page in range(n_pages):
        fig, axes = plt.subplots(2, 2, figsize=(14, 8))
        axes = axes.flatten()

        start = page * per_page
        end = start + per_page
        subset = sample_cols[start:end]

        for i, sample in enumerate(subset):
            ax = axes[i]

            ax.scatter(z_df["genome_pos"], z_df[sample], s=2, alpha=0.5)

            ax.set_ylim(-5, 5)
            ax.axhline(2, linestyle="--", color="gray")
            ax.axhline(-2, linestyle="--", color="gray")

            ax.set_title(sample)
            ax.set_xlabel("Genome")
            ax.set_ylabel("Z-score")

        for j in range(len(subset), 4):
            fig.delaxes(axes[j])

        plt.tight_layout()
        plt.savefig(out / f"grid_page_{page+1}.png", dpi=150)
        plt.close()

In [8]:
plot_in_grids(z_new)